In [ ]:
# ============================================================
# Analyse statistique — Forme linguistique & Confiance envers l'IA
# PIR 2025-2026 — Yousra MEDJIDEL
# ============================================================

import pandas as pd
import numpy as np
import pingouin as pg
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

COLORS = {'A': '#2E86AB', 'B': '#A23B72', 'C': '#F18F01', 'D': '#C73E1D'}
CONDITIONS = ['A', 'B', 'C', 'D']

In [ ]:
# ============================================================
# 1. Chargement et préparation des données
# ============================================================

df_raw = pd.read_excel('RéponsesPIR ConfianceIA.xlsx')

# Lignes vides issues de l'export Google Forms
df_raw = df_raw.dropna(subset=['Timestamp'])
# Entrée corrompue identifiée lors de l'export (erreur Google Sheets)
df_raw = df_raw[df_raw['Code participant '] != '#REF!'].reset_index(drop=True)

N = len(df_raw)
print(f"Participants valides : {N}")

cols = list(df_raw.columns)

# Position du premier item Likert de chaque condition dans le fichier source
COND_START = {'A': 14, 'B': 28, 'C': 42, 'D': 56}

# Les 13 items de chaque condition, dans l'ordre où ils apparaissent :
#  0-3   cognitif (fiabilité, exactitude perçue, compétence, utilité)
#  4-5   affectif (informé, adapté aux besoins)
#  6-8   comportemental (partage, usage, confiance)
#  9-11  forme perçue (clarté, soin, lisibilité)
#  12    score global
ITEM_INDICES = {
    'cognitif':       [0, 1, 2, 3],
    'affectif':       [4, 5],
    'comportemental': [6, 7, 8],
    'forme':          [9, 10, 11],
    'global':         [12],
}

scores = pd.DataFrame()

for cond in CONDITIONS:
    start = COND_START[cond]

    for dim, indices in ITEM_INDICES.items():
        items = [cols[start + i] for i in indices]
        scores[f'{cond}_{dim}'] = df_raw[items].mean(axis=1)

    # score total = moyenne des 12 items (le score global est exclu)
    items_12 = [cols[start + i] for i in range(12)]
    scores[f'{cond}_total'] = df_raw[items_12].mean(axis=1)

print("\nValeurs manquantes par colonne :")
print(scores.isnull().sum())

print("\nMoyennes par condition (score total) :")
for cond in CONDITIONS:
    print(f"  {cond} : M = {scores[f'{cond}_total'].mean():.2f}, "
          f"SD = {scores[f'{cond}_total'].std():.2f}")

In [ ]:
# ============================================================
# 2. Statistiques descriptives par dimension
# ============================================================

DIMENSIONS = ['cognitif', 'affectif', 'comportemental', 'forme', 'global', 'total']

for dim in DIMENSIONS:
    print(f"\n-- {dim.upper()} --")
    print(f"  {'Condition':<12} {'M':>6} {'SD':>6} {'Mdn':>6} {'Min':>6} {'Max':>6}")
    for cond in CONDITIONS:
        col = scores[f'{cond}_{dim}']
        print(f"  {cond:<12} {col.mean():>6.2f} {col.std():>6.2f} "
              f"{col.median():>6.2f} {col.min():>6.2f} {col.max():>6.2f}")

In [ ]:
# ============================================================
# 3. Cohérence interne — Alpha de Cronbach
# ============================================================

print("\nAlpha de Cronbach par condition (12 items) :")
print(f"  {'Condition':<12} {'alpha':>7} {'IC bas':>10} {'IC haut':>10}")
for cond in CONDITIONS:
    start = COND_START[cond]
    items_12 = [cols[start + i] for i in range(12)]
    alpha, ci = pg.cronbach_alpha(df_raw[items_12])
    print(f"  {cond:<12} {alpha:>7.3f} {ci[0]:>10.3f} {ci[1]:>10.3f}")

In [ ]:
# ============================================================
# 4. Vérification des postulats — normalité et homogénéité
# ============================================================

print("\nTest de Shapiro-Wilk (normalité) :")
print(f"  {'Condition':<12} {'W':>8} {'p':>10} {'Normal'}")
for cond in CONDITIONS:
    col = scores[f'{cond}_total'].values
    stat, p = stats.shapiro(col)
    normal = "oui" if p > 0.05 else "non (p<0.05)"
    print(f"  {cond:<12} {stat:>8.4f} {p:>10.4f}  {normal}")

groupes = [scores[f'{c}_total'].values for c in CONDITIONS]
stat_lev, p_lev = stats.levene(*groupes)
homo = "satisfait" if p_lev > 0.05 else "à surveiller"
print(f"\nTest de Levene (homogénéité) : F = {stat_lev:.4f}, p = {p_lev:.4f} -> {homo}")
print("(l'ANOVA à mesures répétées est peu sensible à ce postulat)")

# --- Figure : distributions par condition (histogramme + KDE + normale théorique) ---
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)

SHAPIRO_LABELS = {
    'A': 'Condition A — Soignée + Vraie\nW=0.978, p=0.456',
    'B': 'Condition B — Dégradée + Vraie\nW=0.961, p=0.087',
    'C': 'Condition C — Soignée + Fausse\nW=0.929, p=0.004',
    'D': 'Condition D — Dégradée + Fausse\nW=0.921, p=0.002',
}
positions = [gs[0, 0], gs[0, 1], gs[1, 0], gs[1, 1]]

for cond, pos in zip(CONDITIONS, positions):
    ax = fig.add_subplot(pos)
    data = scores[f'{cond}_total'].values
    color = COLORS[cond]

    ax.hist(data, bins=12, density=True, color=color, alpha=0.45,
            edgecolor='white', linewidth=0.8, label='Distribution observée')

    kde = stats.gaussian_kde(data, bw_method=0.5)
    x_range = np.linspace(1, 7, 300)
    ax.plot(x_range, kde(x_range), color=color, linewidth=2.5, label='Densité (KDE)')

    mu, sigma = data.mean(), data.std()
    normal_curve = stats.norm.pdf(x_range, mu, sigma)
    ax.plot(x_range, normal_curve, color='#333333', linewidth=1.8,
             linestyle='--', alpha=0.7, label='Normale théorique')

    ax.axvline(mu, color=color, linestyle=':', linewidth=1.5, alpha=0.8)
    ax.axvline(4, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.text(mu + 0.1, ax.get_ylim()[1] * 0.92 if ax.get_ylim()[1] > 0 else 0.5,
            f'M={mu:.2f}', color=color, fontsize=9, fontweight='bold')

    ax.set_title(SHAPIRO_LABELS[cond], fontsize=10, fontweight='bold', pad=8)
    ax.set_xlabel('Score total de confiance (1–7)', fontsize=9)
    ax.set_ylabel('Densité', fontsize=9)
    ax.set_xlim(0.5, 7.5)
    ax.set_xticks(range(1, 8))
    ax.legend(fontsize=8, framealpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    "Distribution des scores de confiance par condition\n"
    "Histogramme + densité KDE + courbe normale théorique (N=52)",
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('distributions_shapiro.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 5. ANOVA à mesures répétées 2x2 — Test de H1
# ============================================================

# Format long requis par pingouin : une ligne par observation (sujet x condition)
long_df = pd.DataFrame()
for cond in CONDITIONS:
    tmp = pd.DataFrame({
        'subject':    range(N),
        'condition':  cond,
        'forme':      'Soignée' if cond in ['A', 'C'] else 'Dégradée',
        'exactitude': 'Vraie' if cond in ['A', 'B'] else 'Fausse',
        'score':      scores[f'{cond}_total'].values,
    })
    long_df = pd.concat([long_df, tmp], ignore_index=True)

aov = pg.rm_anova(
    data=long_df,
    dv='score',
    within=['forme', 'exactitude'],
    subject='subject',
    detailed=True,
    effsize='np2'
)

print("\nANOVA à mesures répétées 2x2 — score total de confiance :")
print(aov[['Source', 'F', 'p_unc', 'np2', 'eps']].to_string(index=False))

EFFET_NOMS = {
    'forme':              'Forme linguistique (H1)',
    'exactitude':         'Exactitude du contenu',
    'forme * exactitude': 'Interaction Forme x Exactitude',
}

print("\nInterprétation détaillée :")
for _, row in aov.iterrows():
    src, F, p, eta, eps = row['Source'], row['F'], row['p_unc'], row['np2'], row['eps']
    label = EFFET_NOMS.get(src, src)

    if eta < 0.06:
        taille = "petit"
    elif eta < 0.14:
        taille = "moyen"
    else:
        taille = "grand"

    sig = "significatif" if p < 0.05 else "non significatif"
    print(f"  {label}")
    print(f"    F = {F:.3f} | p = {p:.4f} | eta2p = {eta:.3f} | eps = {eps:.3f}")
    print(f"    -> {sig} | taille d'effet : {taille}")

# --- Figure 1 : graphique d'interaction Forme x Exactitude ---
fig1, ax1 = plt.subplots(figsize=(9, 7))

cellules = {
    'Soignée':  {'Vraie': scores['A_total'], 'Fausse': scores['C_total']},
    'Dégradée': {'Vraie': scores['B_total'], 'Fausse': scores['D_total']},
}
x_pos = [0, 1]

for forme, style, marker, color in zip(
    ['Soignée', 'Dégradée'], ['-', '--'], ['o', 's'], ['#2E86AB', '#C73E1D']
):
    moyennes = [cellules[forme][exact].mean() for exact in ['Vraie', 'Fausse']]
    erreurs  = [cellules[forme][exact].sem() for exact in ['Vraie', 'Fausse']]

    ax1.errorbar(
        x_pos, moyennes, yerr=erreurs, label=f'Forme {forme}',
        color=color, marker=marker, markersize=12, linewidth=2.8,
        linestyle=style, capsize=8, capthick=2,
        markeredgecolor='white', markeredgewidth=2, zorder=3
    )

    for x, m in zip(x_pos, moyennes):
        decalage = 0.08 if forme == 'Soignée' else -0.15
        ax1.annotate(f'{m:.2f}', xy=(x, m), xytext=(x + 0.07, m + decalage),
                      fontsize=12, fontweight='bold', color=color)

ax1.fill_between(
    x_pos,
    [cellules['Soignée']['Vraie'].mean(), cellules['Soignée']['Fausse'].mean()],
    [cellules['Dégradée']['Vraie'].mean(), cellules['Dégradée']['Fausse'].mean()],
    alpha=0.07, color='gray', label='Écart forme (stable)'
)

ax1.set_xlim(-0.3, 1.3)
ax1.set_ylim(2, 6.5)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(['Contenu Vrai', 'Contenu Faux'], fontsize=12)
ax1.set_ylabel('Score moyen de confiance ± ES', fontsize=11)
ax1.set_title('Graphique d\'interaction\nForme linguistique × Exactitude du contenu',
               fontsize=13, fontweight='bold', pad=14)
ax1.axhline(4, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax1.text(1.32, 4.05, 'Neutre\n(4/7)', color='gray', fontsize=9)
ax1.legend(fontsize=11, framealpha=0.25, loc='lower right')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax1.text(
    0.02, 0.97,
    'ANOVA mesures répétées :\n'
    'Forme : F(1,51)=39.96, p<.001, eta2p=.439\n'
    'Exactitude : F(1,51)=0.63, p=.432, eta2p=.012\n'
    'Interaction : F(1,51)=1.52, p=.223, eta2p=.029',
    transform=ax1.transAxes, fontsize=9, verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow',
              edgecolor='#cccc99', alpha=0.85)
)

fig1.suptitle(
    'Effet de la forme linguistique et de l\'exactitude du contenu\n'
    'sur la confiance perçue envers les réponses d\'IA (N = 52)',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('figure1_interaction.png', bbox_inches='tight', dpi=150)
plt.show()

# --- Figure 2 : boîtes à moustaches par condition ---
fig2, ax2 = plt.subplots(figsize=(9, 7))

data_bp = [scores[f'{c}_total'].values for c in CONDITIONS]

bp = ax2.boxplot(
    data_bp, patch_artist=True, notch=False, widths=0.55,
    medianprops={'color': 'white', 'linewidth': 2.8},
    whiskerprops={'linewidth': 1.8, 'linestyle': '--'},
    capprops={'linewidth': 2},
    flierprops={'marker': 'o', 'markersize': 5, 'alpha': 0.5, 'markeredgecolor': 'white'}
)

for patch, cond in zip(bp['boxes'], CONDITIONS):
    patch.set_facecolor(COLORS[cond])
    patch.set_alpha(0.82)
    patch.set_linewidth(1.5)
for flier, cond in zip(bp['fliers'], CONDITIONS):
    flier.set_markerfacecolor(COLORS[cond])

np.random.seed(42)
for i, (data, cond) in enumerate(zip(data_bp, CONDITIONS), start=1):
    jitter = np.random.normal(0, 0.07, size=len(data))
    ax2.scatter(i + jitter, data, color=COLORS[cond], alpha=0.35,
                s=30, zorder=2, edgecolors='white', linewidths=0.4)

ax2.set_xticklabels(['A\nSoignée\n+ Vraie', 'B\nDégradée\n+ Vraie',
                       'C\nSoignée\n+ Fausse', 'D\nDégradée\n+ Fausse'], fontsize=11)
ax2.set_ylim(0.5, 7.5)
ax2.set_yticks(range(1, 8))
ax2.axhline(4, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax2.text(4.6, 4.08, 'Neutre (4/7)', color='gray', fontsize=9)
ax2.set_ylabel('Score total de confiance (1–7)', fontsize=12)
ax2.set_title('Distribution des scores par condition\n'
               '(boîtes à moustaches + points individuels)',
               fontsize=13, fontweight='bold', pad=14)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

legend_patches = [
    mpatches.Patch(color=COLORS[c], alpha=0.82, label=lbl)
    for c, lbl in zip(CONDITIONS, ['A — Soignée+Vraie', 'B — Dégradée+Vraie',
                                     'C — Soignée+Fausse', 'D — Dégradée+Fausse'])
]
ax2.legend(handles=legend_patches, fontsize=10, loc='lower right', framealpha=0.25)

fig2.suptitle(
    'Effet de la forme linguistique et de l\'exactitude du contenu\n'
    'sur la confiance perçue envers les réponses d\'IA (N = 52)',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('figure2_boxplots.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 6. Tests post-hoc — Wilcoxon signé avec correction de Bonferroni
# ============================================================

COMPARAISONS = [
    ('A', 'B', 'Forme | Contenu Vrai (H1)'),
    ('C', 'D', 'Forme | Contenu Faux (H1 bis)'),
    ('A', 'C', 'Exactitude | Forme Soignée (H2)'),
    ('B', 'D', 'Exactitude | Forme Dégradée'),
    ('B', 'C', 'Dégradée+Vraie vs Soignée+Fausse (coeur de H2)'),
    ('A', 'D', 'Soignée+Vraie vs Dégradée+Fausse (extrêmes)'),
]
N_COMP = len(COMPARAISONS)

print(f"\nTests post-hoc — {N_COMP} comparaisons, seuil de Bonferroni = "
      f"0.05/{N_COMP} = {0.05/N_COMP:.4f}")


def cohen_d(x, y):
    diff = np.mean(x) - np.mean(y)
    pooled_sd = np.sqrt((np.std(x, ddof=1)**2 + np.std(y, ddof=1)**2) / 2)
    return diff / pooled_sd


resultats = []
for c1, c2, label in COMPARAISONS:
    x = scores[f'{c1}_total'].values
    y = scores[f'{c2}_total'].values

    stat_w, p_brut = stats.wilcoxon(x, y)
    p_bonf = min(p_brut * N_COMP, 1.0)
    d = cohen_d(x, y)

    abs_d = abs(d)
    if abs_d < 0.2:
        taille = "négligeable"
    elif abs_d < 0.5:
        taille = "petit"
    elif abs_d < 0.8:
        taille = "moyen"
    else:
        taille = "grand"

    if p_bonf < 0.001:
        sig = "p<.001"
    elif p_bonf < 0.01:
        sig = "p<.01"
    elif p_bonf < 0.05:
        sig = "p<.05"
    else:
        sig = "ns"

    resultats.append({
        'Comparaison': label, 'M_c1': np.mean(x), 'M_c2': np.mean(y),
        'delta': np.mean(x) - np.mean(y), 'W': stat_w,
        'p_brut': p_brut, 'p_Bonf': p_bonf, 'd': d,
        'Taille': taille, 'Sig': sig
    })

    print(f"\n{label}")
    print(f"  M({c1})={np.mean(x):.2f} vs M({c2})={np.mean(y):.2f} "
          f"(delta = {np.mean(x)-np.mean(y):+.2f})")
    print(f"  W={stat_w:.0f} | p_brut={p_brut:.4f} | p_Bonf={p_bonf:.4f} | "
          f"d={d:.3f} ({taille}) -> {sig}")

df_res = pd.DataFrame(resultats)
print("\nTableau récapitulatif :")
print(df_res[['Comparaison', 'M_c1', 'M_c2', 'delta', 'p_brut', 'p_Bonf',
              'd', 'Taille', 'Sig']].to_string(index=False))

In [ ]:
# ============================================================
# 7. ANOVA par dimension de la confiance
# ============================================================

DIMENSIONS_LABELS = {
    'cognitif':       'Dimension cognitive',
    'affectif':       'Dimension affective',
    'comportemental': 'Dimension comportementale',
    'forme':          'Forme linguistique perçue',
}

resultats_dim = []

for dim_key, dim_label in DIMENSIONS_LABELS.items():
    print(f"\n-- {dim_label.upper()} --")
    print(f"  {'Condition':<12} {'M':>6} {'SD':>6} {'Mdn':>6}")
    for cond in CONDITIONS:
        col = scores[f'{cond}_{dim_key}']
        print(f"  {cond:<12} {col.mean():>6.2f} {col.std():>6.2f} {col.median():>6.2f}")

    long_dim = pd.DataFrame()
    for cond in CONDITIONS:
        tmp = pd.DataFrame({
            'subject':    range(N),
            'forme':      'Soignée' if cond in ['A', 'C'] else 'Dégradée',
            'exactitude': 'Vraie' if cond in ['A', 'B'] else 'Fausse',
            'score':      scores[f'{cond}_{dim_key}'].values,
        })
        long_dim = pd.concat([long_dim, tmp], ignore_index=True)

    aov_dim = pg.rm_anova(
        data=long_dim, dv='score', within=['forme', 'exactitude'],
        subject='subject', detailed=True, effsize='np2'
    )

    noms_effets = {
        'forme': 'Forme linguistique', 'exactitude': 'Exactitude',
        'forme * exactitude': 'Interaction',
    }

    print(f"\n  ANOVA à mesures répétées :")
    for _, row in aov_dim.iterrows():
        src, F, p, eta = row['Source'], row['F'], row['p_unc'], row['np2']
        label = noms_effets.get(src, src)
        sig = 'p<.001' if p < 0.001 else ('p<.01' if p < 0.01 else ('p<.05' if p < 0.05 else 'ns'))
        taille = 'petit' if eta < 0.06 else ('moyen' if eta < 0.14 else 'grand')
        print(f"  {label:<25} F={F:>7.3f}  p={p:>8.4f}  eta2p={eta:>6.3f}  {sig}")

        if src == 'forme':
            resultats_dim.append({
                'Dimension': dim_label, 'Effet': label,
                'F': F, 'p': p, 'eta2p': eta, 'Taille': taille, 'Sig': sig
            })

    if dim_key != 'forme':
        forme_pool = pd.concat([scores[f'{c}_forme'] for c in CONDITIONS])
        dim_pool   = pd.concat([scores[f'{c}_{dim_key}'] for c in CONDITIONS])
        rs_g, p_g = stats.spearmanr(forme_pool, dim_pool)
        print(f"  Corrélation Spearman (forme perçue <-> {dim_key}, global) : "
              f"rs={rs_g:.3f}, p={p_g:.4f}")

df_recap = pd.DataFrame(resultats_dim)
print("\nRécapitulatif — effet de la forme par dimension :")
print(df_recap[['Dimension', 'F', 'p', 'eta2p', 'Taille', 'Sig']].to_string(index=False))

# --- Figure 3 : scores moyens par dimension et par condition ---
fig3, ax3 = plt.subplots(figsize=(10, 7))

dims_labels = ['Cognitive\n(4 items)', 'Affective\n(2 items)',
               'Comportementale\n(3 items)', 'Forme perçue\n(3 items)']
dims_keys = ['cognitif', 'affectif', 'comportemental', 'forme']

x = np.arange(len(dims_labels))
width = 0.18
cond_labels = {'A': 'A — Soignée+Vraie', 'B': 'B — Dégradée+Vraie',
               'C': 'C — Soignée+Fausse', 'D': 'D — Dégradée+Fausse'}
offsets = [-1.5, -0.5, 0.5, 1.5]

for cond, offset in zip(CONDITIONS, offsets):
    moyennes = [scores[f'{cond}_{d}'].mean() for d in dims_keys]
    erreurs  = [scores[f'{cond}_{d}'].sem() for d in dims_keys]

    bars = ax3.bar(x + offset * width, moyennes, width=width, yerr=erreurs,
                    capsize=4, color=COLORS[cond], alpha=0.82,
                    label=cond_labels[cond],
                    error_kw={'elinewidth': 1.5, 'ecolor': '#333333'})

    for bar, m in zip(bars, moyennes):
        ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.25,
                  f'{m:.2f}', ha='center', va='bottom', fontsize=7.5,
                  fontweight='bold', color=COLORS[cond])

ax3.set_xticks(x)
ax3.set_xticklabels(dims_labels, fontsize=11)
ax3.set_ylim(1, 8)
ax3.set_yticks(range(1, 8))
ax3.set_ylabel('Score moyen ± ES (échelle 1–7)', fontsize=11)
ax3.axhline(4, color='gray', linestyle=':', linewidth=1, alpha=0.6)
ax3.text(3.6, 4.08, 'Neutre (4/7)', color='gray', fontsize=9)
ax3.set_title('Scores moyens par dimension et par condition\n(N = 52, échelle 1–7)',
               fontsize=13, fontweight='bold', pad=14)
ax3.legend(fontsize=10, framealpha=0.25, loc='upper right', ncol=2)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

eta_values = [0.335, 0.425, 0.203, 0.516]
for i, eta in enumerate(eta_values):
    ax3.text(i, 1.15, f'eta2p = {eta:.3f}', ha='center', fontsize=9,
              color='#555555', fontstyle='italic')

plt.tight_layout()
plt.savefig('figure3_dimensions_barres.png', bbox_inches='tight', dpi=150)
plt.show()

# --- Figure 4 : gradient de l'effet de forme + corrélations Spearman ---
fig4, axes4 = plt.subplots(1, 2, figsize=(13, 6))

ax4a = axes4[0]
dims_noms = ['Forme\nperçue', 'Affective', 'Cognitive', 'Comportementale']
eta_vals = [0.516, 0.425, 0.335, 0.203]
F_vals   = [54.30, 37.67, 25.66, 12.96]
colors_grad = ['#1a5276', '#2E86AB', '#5dade2', '#aed6f1']

bars4a = ax4a.barh(dims_noms, eta_vals, color=colors_grad, alpha=0.88,
                    height=0.55, edgecolor='white', linewidth=1)

for seuil in [0.01, 0.06, 0.14]:
    ax4a.axvline(seuil, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax4a.text(0.01, 4.1, 'petit', color='gray', fontsize=8, ha='center')
ax4a.text(0.06, 4.1, 'moyen', color='gray', fontsize=8, ha='center')
ax4a.text(0.14, 4.1, 'grand', color='gray', fontsize=8, ha='center')

for bar, eta, F, color in zip(bars4a, eta_vals, F_vals, colors_grad):
    ax4a.text(eta + 0.01, bar.get_y() + bar.get_height() / 2,
              f'eta2p = {eta:.3f}  |  F = {F:.2f}***',
              va='center', fontsize=10, fontweight='bold', color=color)

ax4a.set_xlim(0, 0.75)
ax4a.set_xlabel('Eta-carré partiel (eta2p)', fontsize=11)
ax4a.set_title('Effet de la forme linguistique\npar dimension (eta2p)',
                fontsize=12, fontweight='bold', pad=12)
ax4a.spines['top'].set_visible(False)
ax4a.spines['right'].set_visible(False)

ax4b = axes4[1]
dims_corr = ['Affective', 'Comportementale', 'Cognitive']
rs_vals = [0.667, 0.625, 0.630]
colors_corr = ['#2E86AB', '#F18F01', '#A23B72']

bars4b = ax4b.barh(dims_corr, rs_vals, color=colors_corr, alpha=0.85,
                    height=0.45, edgecolor='white')

for seuil in [0.3, 0.5, 0.7]:
    ax4b.axvline(seuil, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax4b.text(0.3, 3.25, 'faible', color='gray', fontsize=8, ha='center')
ax4b.text(0.5, 3.25, 'modérée', color='gray', fontsize=8, ha='center')
ax4b.text(0.7, 3.25, 'forte', color='gray', fontsize=8, ha='center')

for bar, rs, color in zip(bars4b, rs_vals, colors_corr):
    ax4b.text(rs + 0.01, bar.get_y() + bar.get_height() / 2,
              f'rs = {rs:.3f}***', va='center', fontsize=10,
              fontweight='bold', color=color)

ax4b.set_xlim(0, 0.9)
ax4b.set_xlabel('Corrélation de Spearman (rs)', fontsize=11)
ax4b.set_title('Corrélation forme perçue <-> dimension\n(toutes conditions poolées)',
                fontsize=12, fontweight='bold', pad=12)
ax4b.spines['top'].set_visible(False)
ax4b.spines['right'].set_visible(False)

fig4.suptitle(
    'Analyse par dimensions — Effet de la forme linguistique\n'
    'sur les composantes de la confiance perçue (N = 52)',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('figure4_gradient_dimensions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 8. Items méta-réflexifs, classement et détection des erreurs
# ============================================================

# -- Items méta-réflexifs --
meta_items = {
    'Q1 — Qualité écriture -> confiance':    cols[76],
    'Q2 — Bien formulée = plus vraie':        cols[77],
    'Q3 — Mal à dissocier style/véracité':    cols[78],
    "Q4 — Bien s'exprimer = plus compétent":  cols[79],
}

print(f"\nItems méta-réflexifs :")
print(f"  {'Item':<42} {'M':>5} {'SD':>5} {'Mdn':>5} {'%>=5':>6} {'%<=3':>6}")
for label, col in meta_items.items():
    vals = df_raw[col].dropna()
    m, sd, mdn = vals.mean(), vals.std(), vals.median()
    pct_accord = (vals >= 5).sum() / len(vals) * 100
    pct_desaccord = (vals <= 3).sum() / len(vals) * 100
    stat_w, p_w = stats.wilcoxon(vals - 4)
    sig = 'p<.001' if p_w < 0.001 else ('p<.05' if p_w < 0.05 else 'ns')
    print(f"  {label:<42} {m:>5.2f} {sd:>5.2f} {mdn:>5.1f} "
          f"{pct_accord:>5.1f} {pct_desaccord:>5.1f}  [{sig}]")

# -- Classement comparatif --
rang_cols = {
    '1ère place': cols[70], '2ème place': cols[71],
    '3ème place': cols[72], '4ème place': cols[73],
}

print(f"\nDistribution des votes par rang :")
for rang, col in rang_cols.items():
    dist = df_raw[col].value_counts()
    votes = {c: dist.get(f'Réponse {c}', 0) for c in CONDITIONS}
    total = sum(votes.values())
    print(f"  {rang:<15} " + "  ".join(
        f"{c}:{votes[c]}({votes[c]/total*100:.0f}%)" for c in CONDITIONS
    ))

print(f"\nRang moyen par condition (1 = plus fiable, 4 = moins fiable) :")
for cond in CONDITIONS:
    score_rang = 0
    for rang_num, (rang, col) in enumerate(rang_cols.items(), start=1):
        dist = df_raw[col].value_counts()
        votes = dist.get(f'Réponse {cond}', 0)
        score_rang += rang_num * votes
    print(f"  {cond} : {score_rang / N:.2f}")

# -- Détection des erreurs factuelles --
detection_col = df_raw[cols[75]]
detecte_C = detection_col.str.contains('Réponse C', na=False)
detecte_D = detection_col.str.contains('Réponse D', na=False)
detecte_rien = detection_col.str.contains('Non', na=False) & ~detection_col.str.contains('Oui', na=False)
incertain = detection_col.str.contains('sûr', na=False)

print(f"\nDétection des erreurs factuelles :")
print(f"  Condition C détectée comme fausse : {detecte_C.sum()}/{N} ({detecte_C.mean()*100:.1f}%)")
print(f"  Condition D détectée comme fausse : {detecte_D.sum()}/{N} ({detecte_D.mean()*100:.1f}%)")
print(f"  Aucune erreur détectée            : {detecte_rien.sum()}/{N} ({detecte_rien.mean()*100:.1f}%)")
print(f"  Incertain·e                       : {incertain.sum()}/{N} ({incertain.mean()*100:.1f}%)")

# -- Prise de conscience des biais --
conscience_col = df_raw[cols[81]].dropna()
oui = conscience_col.isin(['Oui, beaucoup', 'Plutôt oui']).sum()
print(f"\nPrise de conscience des biais : {oui}/{N} ({oui/N*100:.1f}%) "
      f"déclarent 'Oui beaucoup' ou 'Plutôt oui'")

# --- Figure 5 : items méta-réflexifs ---
fig5, ax5 = plt.subplots(figsize=(12, 9))

meta_labels = [
    "Q2 — Une réponse bien formulée\nm'a semblé plus vraie",
    "Q1 — La qualité d'écriture\na influencé ma confiance",
    "Q4 — Bien s'exprimer\n= plus compétent",
    "Q3 — Difficile de dissocier\nstyle et véracité",
]
meta_means = [5.02, 4.75, 4.71, 4.02]
meta_sems = [df_raw[cols[77]].sem(), df_raw[cols[76]].sem(),
             df_raw[cols[79]].sem(), df_raw[cols[78]].sem()]
meta_pcts_accord = [65.4, 67.3, 59.6, 48.1]
meta_pcts_desaccord = [21.2, 26.9, 28.8, 36.5]
meta_sig = ['*** p<.001', '* p<.05', '* p<.05', 'ns']
meta_colors = ['#1a5276', '#2E86AB', '#5dade2', '#aed6f1']
y_pos = np.array([0, 2.5, 5.0, 7.5])

bars5 = ax5.barh(y_pos, meta_means, xerr=meta_sems, color=meta_colors,
                  alpha=0.88, height=1.0, edgecolor='white', linewidth=1,
                  error_kw={'elinewidth': 1.8, 'ecolor': '#333333', 'capsize': 5})

ax5.axvline(4, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
ax5.set_xlim(0, 7)
ax5.set_ylim(-1.5, 9.5)

for i, (bar, m, sig, pa, pd) in enumerate(
    zip(bars5, meta_means, meta_sig, meta_pcts_accord, meta_pcts_desaccord)
):
    y_centre = bar.get_y() + bar.get_height() / 2
    ax5.annotate(f'M = {m:.2f}   {sig}', xy=(m, y_centre), xytext=(6.95, y_centre),
                  fontsize=10, fontweight='bold', color=meta_colors[i],
                  va='center', ha='left', annotation_clip=False)
    ax5.text(2.0, bar.get_y() - 0.65,
              f'Accord (≥5) : {pa}%       Désaccord (≤3) : {pd}%',
              va='top', ha='left', fontsize=9, color='#666666', fontstyle='italic')

ax5.set_yticks(y_pos)
ax5.set_yticklabels(meta_labels, fontsize=10.5)
ax5.set_xticks(range(1, 8))
ax5.set_xlabel('Score moyen ± ES (échelle 1–7)', fontsize=11)
ax5.set_title("Items méta-réflexifs — Conscience de l'influence\n"
               "de la forme linguistique sur la confiance (N = 52)",
               fontsize=13, fontweight='bold', pad=16)
ax5.text(4.05, 9.2, 'Point neutre (4/7)', color='gray', fontsize=9, va='top')
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('figure5_meta_reflexifs.png', bbox_inches='tight', dpi=150)
plt.show()

# --- Figure 6 : classement comparatif ---
fig6, axes6 = plt.subplots(1, 2, figsize=(16, 7))

ax6a = axes6[0]
rangs = ['1ère\n(+ fiable)', '2ème', '3ème', '4ème\n(− fiable)']
votes = {'A': [24, 15, 8, 6], 'B': [13, 10, 15, 15],
         'C': [10, 22, 8, 12], 'D': [5, 5, 21, 19]}
pcts = {'A': [46, 29, 15, 12], 'B': [25, 19, 29, 29],
        'C': [19, 42, 15, 23], 'D': [10, 10, 40, 37]}

x6 = np.arange(len(rangs))
w6 = 0.19
offs = [-1.5, -0.5, 0.5, 1.5]

for cond, offset in zip(CONDITIONS, offs):
    bars_r = ax6a.bar(x6 + offset * w6, votes[cond], width=w6,
                       color=COLORS[cond], alpha=0.85, edgecolor='white')
    for bar, v, p in zip(bars_r, votes[cond], pcts[cond]):
        ax6a.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                   f'{v}', ha='center', va='bottom', fontsize=9.5,
                   fontweight='bold', color='#222222')
        ax6a.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2.4,
                   f'({p}%)', ha='center', va='bottom', fontsize=8, color='#666666')

ax6a.set_xticks(x6)
ax6a.set_xticklabels(rangs, fontsize=11)
ax6a.set_ylabel('Nombre de votes (N = 52)', fontsize=11)
ax6a.set_ylim(0, 40)
ax6a.set_title('Distribution des votes par rang\net par condition',
                fontsize=12, fontweight='bold', pad=12)

legend_patches = [
    mpatches.Patch(color=COLORS[c], alpha=0.85,
                   label=f'{c} — ' + {'A': 'Soignée+Vraie', 'B': 'Dégradée+Vraie',
                                        'C': 'Soignée+Fausse', 'D': 'Dégradée+Fausse'}[c])
    for c in CONDITIONS
]
ax6a.legend(handles=legend_patches, fontsize=9, framealpha=0.25, loc='upper right')
ax6a.spines['top'].set_visible(False)
ax6a.spines['right'].set_visible(False)

ax6b = axes6[1]
conds_r = ['A', 'C', 'B', 'D']
rangs_m = [1.96, 2.42, 2.65, 2.96]
labels_r = ['A — Soignée + Vraie', 'C — Soignée + Fausse',
            'B — Dégradée + Vraie', 'D — Dégradée + Fausse']
y_rang = np.array([0, 1.5, 3.0, 4.5])

bars6b = ax6b.barh(y_rang, rangs_m, color=[COLORS[c] for c in conds_r],
                    alpha=0.85, height=0.7, edgecolor='white')

ax6b.axvline(2.5, color='gray', linestyle='--', linewidth=1.2, alpha=0.6)
ax6b.text(2.52, 5.1, 'Milieu (2.5)', color='gray', fontsize=9)

for bar, rm, cond in zip(bars6b, rangs_m, conds_r):
    y_c = bar.get_y() + bar.get_height() / 2
    ax6b.text(rm - 0.07, y_c, f'{rm:.2f}', va='center', ha='right',
               fontsize=11, fontweight='bold', color=COLORS[cond])

ax6b.set_yticks(y_rang)
ax6b.set_yticklabels(labels_r, fontsize=10.5)
ax6b.set_xlim(3.5, 1)
ax6b.set_xticks([1, 1.5, 2, 2.5, 3, 3.5])
ax6b.set_ylim(-0.8, 5.8)
ax6b.set_xlabel('Rang moyen (1 = + fiable, 4 = − fiable)', fontsize=11)
ax6b.set_title('Rang moyen par condition\n(classement comparatif)',
                fontsize=12, fontweight='bold', pad=12)
ax6b.spines['top'].set_visible(False)
ax6b.spines['right'].set_visible(False)

fig6.suptitle('Classement comparatif des quatre réponses (N = 52)\n'
              'Convergence avec les scores Likert : A > C > B > D',
              fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figure6_classement.png', bbox_inches='tight', dpi=150)
plt.show()

# --- Figure 7 : détection des erreurs + prise de conscience ---
fig7, axes7 = plt.subplots(1, 2, figsize=(15, 7))

ax7a = axes7[0]
det_labels = ["Incertain·e seulement", "Aucune erreur détectée",
              "A détecté D\n(Dégradée+Fausse)", "A détecté C\n(Soignée+Fausse)",
              "A détecté A\n(vraie, par erreur)"]
det_vals = [44.2, 11.5, 21.2, 15.4, 9.6]
det_colors = ['#95a5a6', '#27ae60', '#e67e22', '#e74c3c', '#8e44ad']
y_det = np.array([0, 1.8, 3.6, 5.4, 7.2])

bars7a = ax7a.barh(y_det, det_vals, color=det_colors, alpha=0.85,
                    height=0.9, edgecolor='white')

ax7a.set_xlim(0, 65)
for bar, v, color in zip(bars7a, det_vals, det_colors):
    ax7a.text(v + 1.2, bar.get_y() + bar.get_height() / 2, f'{v}%',
               va='center', ha='left', fontsize=11, fontweight='bold', color=color)

ax7a.set_yticks(y_det)
ax7a.set_yticklabels(det_labels, fontsize=10.5)
ax7a.set_ylim(-1.2, 8.8)
ax7a.set_xlabel('Pourcentage de participants (%)', fontsize=11)
ax7a.set_title('Détection des erreurs factuelles\n(réponses multiples possibles)',
                fontsize=12, fontweight='bold', pad=12)

ax7a.annotate(
    "Seulement 15.4%\ndétectent l'inexactitude\nde C (bien écrite\nmais fausse)",
    xy=(15.4, 5.4), xytext=(38, 6.5), fontsize=9, color='#e74c3c',
    arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5),
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#fdecea',
              edgecolor='#e74c3c', alpha=0.9)
)
ax7a.spines['top'].set_visible(False)
ax7a.spines['right'].set_visible(False)

ax7b = axes7[1]
con_labels = ['Oui,\nbeaucoup', 'Plutôt\noui', 'Plutôt\nnon', 'Non']
con_vals = [14, 29, 7, 2]
con_pcts = [26.9, 55.8, 13.5, 3.8]
con_colors = ['#1a5276', '#2E86AB', '#e67e22', '#e74c3c']

x_con = np.arange(len(con_labels))
bars7b = ax7b.bar(x_con, con_vals, color=con_colors, alpha=0.85,
                   width=0.55, edgecolor='white')

ax7b.set_ylim(0, 44)
for bar, v, pct in zip(bars7b, con_vals, con_pcts):
    ax7b.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.6,
               str(v), ha='center', va='bottom', fontsize=12,
               fontweight='bold', color='#333333')
    ax7b.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2.5,
               f'({pct}%)', ha='center', va='bottom', fontsize=9.5, color='#555555')

ax7b.set_xticks(x_con)
ax7b.set_xticklabels(con_labels, fontsize=11)
ax7b.set_ylabel('Nombre de participants', fontsize=11)
ax7b.set_title('Prise de conscience des biais\naprès le questionnaire',
                fontsize=12, fontweight='bold', pad=12)

ax7b.text(0.5, 0.95, "82.7% ont pris conscience\nde leurs biais d'évaluation",
          transform=ax7b.transAxes, ha='center', va='top', fontsize=11,
          fontweight='bold', color='#1a5276',
          bbox=dict(boxstyle='round,pad=0.5', facecolor='#d6eaf8',
                    edgecolor='#1a5276', alpha=0.9))
ax7b.spines['top'].set_visible(False)
ax7b.spines['right'].set_visible(False)

fig7.suptitle('Vigilance factuelle et prise de conscience des biais (N = 52)',
               fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figure7_detection_conscience.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 9. Données complémentaires — fréquence d'utilisation de l'IA
# ============================================================

freq_col = df_raw["À quelle fréquence utilisez-vous des outils d'IA (ChatGPT, Gemini, Copilot…) ?"]
print("\nFréquence d'utilisation des outils d'IA :")
print(freq_col.value_counts())
print()
print(freq_col.value_counts(normalize=True).mul(100).round(1))